# CSE-CIC-IDS2018 - Preprocessing Pipeline for LSTM IDS

Clean, research-backed preprocessing for the full CSE-CIC-IDS2018 dataset (10 CSVs, ~16M flows).

**Output:** per-flow feature tensors saved as `.npy` for fast loading in the training notebook.

## Design decisions (fixing the old pipeline)

The previous pipeline built **sliding LSTM windows over row-shuffled data** — adjacent rows had no temporal relationship, so the LSTM had nothing real to learn and hallucinated. Published implementations (Ramyaelineni 2020, Wang 2023, Gocs & Johanyak 2024) classify **one flow at a time**, reshaping the feature vector as a `(timesteps=1, features=N)` pseudo-sequence for the LSTM. That is the approach this pipeline follows.

## Pipeline steps

1. **Load** all 10 CSVs in chunks (RAM-safe).
2. **Drop metadata** columns (Timestamp, Dst Port, Protocol) to avoid leakage.
3. **Drop known zero-variance** CICFlowMeter columns (10 columns).
4. **Coerce numerics**, drop Inf/NaN, drop negative durations, drop duplicates.
5. **Normalize labels** (merge typos like `Infilteration -> Infiltration`).
6. **Cap rows per class** for RAM + balance (benign kept larger).
7. **Drop highly correlated** features (|r| > 0.98).
8. **Winsorize** outliers at 1st/99th percentile.
9. **Stratified 70/15/15 split** (train/val/test).
10. **StandardScaler** fit on train only; transform val/test.
11. **Save** to `lstm/processed/`.

## 0 - Imports & seed

In [ ]:
import os, gc, json, time, pickle, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1 - Configuration

In [ ]:
# ---- Paths ----
NOTEBOOK_DIR = Path.cwd()               # .../ai_detection/notebooks
PROJECT_DIR  = NOTEBOOK_DIR.parent      # .../ai_detection
DATA_DIR     = PROJECT_DIR.parent       # .../data  (contains the 10 CSVs)
OUT_DIR      = NOTEBOOK_DIR / 'processed'
OUT_DIR.mkdir(exist_ok=True)

csv_files = sorted(DATA_DIR.glob('*TrafficForML_CICFlowMeter.csv'))
print(f'Found {len(csv_files)} CSV files in {DATA_DIR}')
for p in csv_files:
    print(f'  - {p.name}')
assert len(csv_files) >= 10, 'Expected 10 CSVs'

# ---- Pipeline hyper-parameters ----
CHUNK_SIZE          = 200_000   # rows per read_csv chunk
MAX_ROWS_PER_CLASS  = 200_000   # cap non-benign classes
BENIGN_CAP          = 400_000   # cap benign (kept bigger: it is realistic noise)
MIN_CLASS_ROWS      = 50        # drop classes below this threshold
CORR_THRESHOLD      = 0.98      # |r| above -> drop redundant feature

print(f'\nCaps: benign={BENIGN_CAP:,}, attack={MAX_ROWS_PER_CLASS:,}')
print(f'Output directory: {OUT_DIR}')

## 2 - Column & label maps

In [ ]:
# Metadata / identifier columns that cause leakage or are non-numeric
DROP_META = {'Timestamp', 'Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Protocol'}

# 10 zero-variance columns confirmed in literature (Gocs & Johanyak 2024)
ZERO_VAR_COLS = {
    'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count',
    'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg',
}

# Canonical label names (merge typos & alternate spellings)
LABEL_MAP = {
    'Benign': 'Benign', 'benign': 'Benign', 'BENIGN': 'Benign',
    'FTP-BruteForce': 'FTP-BruteForce',
    'SSH-Bruteforce': 'SSH-BruteForce', 'SSH-BruteForce': 'SSH-BruteForce',
    'DoS attacks-GoldenEye': 'DoS-GoldenEye',
    'DoS attacks-Slowloris': 'DoS-Slowloris',
    'DoS attacks-Hulk': 'DoS-Hulk',
    'DoS attacks-SlowHTTPTest': 'DoS-SlowHTTPTest',
    'DDoS attacks-LOIC-HTTP': 'DDoS-LOIC-HTTP',
    'DDOS attack-HOIC': 'DDoS-HOIC',  'DDoS attack-HOIC': 'DDoS-HOIC',
    'DDOS attack-LOIC-UDP': 'DDoS-LOIC-UDP', 'DDoS attack-LOIC-UDP': 'DDoS-LOIC-UDP',
    'Brute Force -Web': 'BruteForce-Web',
    'Brute Force -XSS': 'BruteForce-XSS',
    'SQL Injection':    'SQL-Injection',
    'Infilteration':    'Infiltration', 'Infiltration': 'Infiltration',
    'Bot': 'Botnet',
}

print(f'Meta dropped : {len(DROP_META)}')
print(f'Zero-var dropped: {len(ZERO_VAR_COLS)}')
print(f'Label classes mapped: {len(set(LABEL_MAP.values()))}')

## 3 - Cleaning functions

In [ ]:
def detect_feature_columns(files):
    """Return the intersection of numeric columns across all CSVs, minus metadata."""
    common = None
    for f in files:
        cols = [c.strip() for c in pd.read_csv(f, nrows=0).columns]
        common = set(cols) if common is None else (common & set(cols))
    if 'Label' not in common:
        raise ValueError('Label column missing in at least one CSV')
    keep = sorted(c for c in common if c not in DROP_META and c not in ZERO_VAR_COLS and c != 'Label')
    keep.append('Label')
    return keep


def clean_chunk(df):
    """Clean one chunk in place: normalize labels, coerce numerics, drop bad rows."""
    df.columns = [c.strip() for c in df.columns]
    df['Label'] = df['Label'].astype(str).str.strip().map(LABEL_MAP)
    df = df.dropna(subset=['Label'])

    feat_cols = [c for c in df.columns if c != 'Label']
    df[feat_cols] = df[feat_cols].apply(pd.to_numeric, errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if 'Flow Duration' in df.columns:
        df = df[df['Flow Duration'] >= 0]

    df = df.drop_duplicates()
    df[feat_cols] = df[feat_cols].astype(np.float32)
    return df


def load_balanced(files, use_cols, chunk_size, attack_cap, benign_cap):
    """Stream CSVs, clean each chunk, and keep a capped buffer per class."""
    buffers = {}
    for i, f in enumerate(files, 1):
        t0, kept = time.time(), 0
        for chunk in pd.read_csv(f, usecols=use_cols, chunksize=chunk_size, low_memory=False):
            cleaned = clean_chunk(chunk)
            if cleaned.empty:
                continue
            kept += len(cleaned)
            for lbl, grp in cleaned.groupby('Label'):
                cap = benign_cap if lbl == 'Benign' else attack_cap
                if lbl not in buffers:
                    buffers[lbl] = grp
                else:
                    buffers[lbl] = pd.concat([buffers[lbl], grp], ignore_index=True)
                if len(buffers[lbl]) > cap:
                    buffers[lbl] = buffers[lbl].sample(n=cap, random_state=SEED).reset_index(drop=True)
            del cleaned
            gc.collect()
        print(f'  [{i:02d}/{len(files)}] {f.name:55s} kept={kept:>9,}  ({time.time()-t0:5.1f}s)')

    # Drop rare classes
    rare = [k for k, v in buffers.items() if len(v) < MIN_CLASS_ROWS]
    for k in rare:
        print(f'  - dropping rare class "{k}" ({len(buffers[k])} rows)')
        del buffers[k]

    return buffers

## 4 - Load & clean the full dataset

In [ ]:
use_cols = detect_feature_columns(csv_files)
print(f'Feature columns kept: {len(use_cols) - 1}   (+ Label)\n')

print('Loading & cleaning all CSVs in chunks...')
buffers = load_balanced(csv_files, use_cols, CHUNK_SIZE, MAX_ROWS_PER_CLASS, BENIGN_CAP)

print('\n=== Class distribution after capping ===')
for lbl in sorted(buffers, key=lambda k: -len(buffers[k])):
    print(f'  {lbl:22s} {len(buffers[lbl]):>9,}')

df = pd.concat(buffers.values(), ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
del buffers; gc.collect()
print(f'\nCombined dataset: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'RAM: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

## 5 - Visualize raw class distribution

In [ ]:
counts = df['Label'].value_counts()
fig, ax = plt.subplots(figsize=(11, 5))
counts.plot.bar(color='steelblue', ax=ax)
ax.set_title('Class distribution after capping (log scale)')
ax.set_yscale('log')
ax.set_ylabel('# flows (log)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'class_distribution.png', dpi=120)
plt.show()

## 6 - Feature cleanup (zero-var + correlation + winsorize)

In [ ]:
X_df   = df.drop(columns=['Label'])
y_text = df['Label'].astype(str)

# Drop remaining zero-variance columns
std = X_df.std()
zero_var = std[std == 0].index.tolist()
if zero_var:
    print(f'Zero-variance dropped: {zero_var}')
    X_df = X_df.drop(columns=zero_var)

# Drop highly correlated features
corr = X_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
to_drop = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]
if to_drop:
    print(f'Highly correlated dropped ({len(to_drop)}): {to_drop}')
    X_df = X_df.drop(columns=to_drop)

# Winsorize outliers at 1/99 percentile per feature
lo = X_df.quantile(0.01)
hi = X_df.quantile(0.99)
X_df = X_df.clip(lower=lo, upper=hi, axis=1)

feature_names = X_df.columns.tolist()
print(f'\nFinal feature count: {len(feature_names)}')
print(f'Samples            : {len(X_df):,}')

## 7 - Encode labels & stratified split

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = label_encoder.classes_.tolist()
num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')

X = X_df.values.astype(np.float32)
del df, X_df, y_text; gc.collect()

# Stratified 70 / 15 / 15
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
del X, X_tmp, y_tmp; gc.collect()

print(f'Train: {X_train.shape}  {dict(Counter(y_train))}')
print(f'Val  : {X_val.shape}')
print(f'Test : {X_test.shape}')

## 8 - Z-score scaling (fit on train only)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# Final sanity: replace any residual NaN/Inf (shouldn't happen but be safe)
for arr in (X_train, X_val, X_test):
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Scaled train mean={X_train.mean():+.4f}  std={X_train.std():.4f}')
print(f'Scaled val   mean={X_val.mean():+.4f}  std={X_val.std():.4f}')

## 9 - Class weights (for weighted loss in training)

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train)
class_weights = class_weights.astype(np.float32)
print('Class weights:')
for name, w in zip(class_names, class_weights):
    print(f'  {name:22s} {w:7.3f}')

## 10 - Save artifacts

In [ ]:
np.save(OUT_DIR / 'X_train.npy', X_train)
np.save(OUT_DIR / 'X_val.npy',   X_val)
np.save(OUT_DIR / 'X_test.npy',  X_test)
np.save(OUT_DIR / 'y_train.npy', y_train)
np.save(OUT_DIR / 'y_val.npy',   y_val)
np.save(OUT_DIR / 'y_test.npy',  y_test)
np.save(OUT_DIR / 'class_weights.npy', class_weights)

with open(OUT_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(OUT_DIR / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
with open(OUT_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)
with open(OUT_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

print(f'Saved artifacts to: {OUT_DIR}')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  - {p.name:25s} {p.stat().st_size/1e6:8.2f} MB')

## Done

Processed tensors are written to `lstm/processed/`. Open `02_training.ipynb` next to train the LSTM.